# Function 8: High-dimensional Optimisation
You’ve reach the final, 8-dimensional search space. High-dimensional black-box optimisation can be very difficult, so sticking to local solutions is not the worst idea here.

In [1]:
import numpy as np
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, WhiteKernel, ConstantKernel
from scipy.optimize import minimize

In [2]:
def load_inputs(file_path):
    with open(file_path, "r") as f:
        content = f.read()

    # Make the file a proper list of lists
    content = "[" + content.replace("]\n[", "],[") + "]"

    # Safe eval with restricted globals
    return eval(content, {"array": np.array})


def load_outputs(file_path):
    with open(file_path, "r") as f:
        content = f.read()

    content = "[" + content.replace("]\n[", "],[") + "]"

    return eval(content, {"np": np})


def get_input_points(function_number, file_path="../inputs.txt"):
    if not 1 <= function_number <= 8:
        raise ValueError("Function number must be between 1 and 8")

    data = load_inputs(file_path)
    index = function_number - 1

    inputs = [dataset[index] for dataset in data]
    return np.array(inputs)


def get_output_points(function_number, file_path="../outputs.txt"):
    if not 1 <= function_number <= 8:
        raise ValueError("Function number must be between 1 and 8")

    data = load_outputs(file_path)
    index = function_number - 1

    outputs = [row[index] for row in data]
    return np.array(outputs)

# Load inputs
X = np.load(r'initial_inputs.npy')
y = np.load(r'initial_outputs.npy')


# Get input and outputs from submissions
inputs_array = get_input_points(8)
outputs_array = get_output_points(8)


# Append inputs_f1_array to X
X = np.vstack((X, inputs_array))

# Append outputs_f1_array to Y
y = np.hstack((y, outputs_array))

print("New shape of X:", X.shape)
print("New shape of Y:", y.shape)

New shape of X: (49, 8)
New shape of Y: (49,)


In [3]:
def generate_nd_grid(max_points, dimensions):
    # define range for input
	r_min, r_max = 0, 1.0

	# generate a random sample from the domain (dimensions)
	nd_grid = r_min + np.random.rand(max_points, dimensions) * (r_max - r_min)

	return np.array(nd_grid)

In [4]:
max_points = 800000
dimensions = 8  # Change this to the desired number of dimensions
X_grid = []
X_grid = generate_nd_grid(max_points, dimensions)

In [5]:
kernel = ConstantKernel(1.0, (1e-2, 1e2)) * Matern(
    length_scale=[0.1]*8,
    length_scale_bounds=(1e-2, 10.0),  # larger upper bound
    nu=2.5
) + WhiteKernel(noise_level=1e-9, noise_level_bounds=(1e-10, 1e-1))

gp = GaussianProcessRegressor(
    kernel=kernel,
    alpha=0.0,
    normalize_y=True,
    n_restarts_optimizer=30,  # more restarts
    random_state=42
)

gp.fit(X, y)

# --- Current best point ---
best_idx = np.argmax(y)
x_best = X[best_idx]

# --- Very tight bounds for final refinement ---
bounds = [(max(0, xi-0.002), min(1, xi+0.002)) for xi in x_best]

# --- Objective: maximize GP posterior mean ---
def neg_gp_mean(x):
    x = np.array(x).reshape(1, -1)
    mu, _ = gp.predict(x, return_std=True)
    return -mu[0]

# --- Multi-start local optimization ---
best_val = -np.inf
best_x = None
for _ in range(40):  # many random starts for robustness
    x0 = np.random.uniform([b[0] for b in bounds], [b[1] for b in bounds])
    res = minimize(
        neg_gp_mean,
        x0=x0,
        bounds=bounds,
        method='L-BFGS-B',
        options={'maxiter':2000}  # increase iterations
    )
    if -res.fun > best_val:
        best_val = -res.fun
        best_x = res.x


# --- Round to 6 decimals ---
next_point = np.round(best_x, 6)

formatted_next_query = f"{next_point[0]:.6f}-{next_point[1]:.6f}-{next_point[2]:.6f}-{next_point[3]:.6f}-{next_point[4]:.6f}-{next_point[5]:.6f}-{next_point[6]:.6f}-{next_point[7]:.6f}"
print("Next point to evaluate (unscaled):", formatted_next_query)

/opt/anaconda3/lib/python3.12/site-packages/sklearn/gaussian_process/_gpr.py:659: ConvergenceWarning: lbfgs failed to converge (status=2):
ABNORMAL_TERMINATION_IN_LNSRCH.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  _check_optimize_result("lbfgs", opt_res)


Next point to evaluate (unscaled): 0.058447-0.067956-0.024929-0.040786-0.405935-0.799055-0.486307-0.891085


/opt/anaconda3/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceWarning: The optimal value found for dimension 4 of parameter k1__k2__length_scale is close to the specified upper bound 10.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceWarning: The optimal value found for dimension 7 of parameter k1__k2__length_scale is close to the specified upper bound 10.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:442: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-10. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
